# 01_langgraph — AOA Actions as LangGraph nodes

`LangGraphAdapter` bridges AOA and LangGraph: register Actions with `.node()`,
declare edges with `.edge()`, `.conditional_edge()`, or `.route()`,
then call `.compile()` — you get a standard LangGraph `CompiledGraph`.

**What’s new**

| Concept | Description |
|---------|-------------|
| `.node(action)` | Register an AOA Action as a LangGraph node |
| `.node(fn, name=...)` | Register a plain async function as a node |
| `.route(src, on=fn, paths={...})` | Route to one of several nodes based on a key |
| `.edge(src, dst)` | Declare a direct edge; unregistered names raise `UnregisteredNodeError` immediately |
| `.build_graph()` | Return an uncompiled `StateGraph` for native LangGraph continuation |
| `.compile()` | Build, validate, and compile the graph |

In [ ]:
!pip install "aoa-langgraph-adapter" langgraph

In [ ]:
import asyncio
from typing import TypedDict

from langgraph.graph import END
from pydantic import Field

from aoa.action_machine.auth import GuestRole
from aoa.action_machine.context import Context
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine
from aoa.langgraph import LangGraphAdapter, UnregisteredNodeError

## State, domain, params, results

The `TypedDict` becomes the LangGraph state schema. Each Action node receives
the full state and returns a partial dict — only the returned keys are merged back.

In [ ]:
class TicketState(TypedDict):
    ticket_id: str
    category: str      # filled by ClassifyTicketAction
    resolved: bool
    note: str


class SupportDomain(BaseDomain):
    name = "support"
    description = "Support ticket processing domain"


class ClassifyParams(BaseParams):
    ticket_id: str = Field(description="Ticket identifier")
    note: str = Field(default="", description="Ticket description")


class ClassifyResult(BaseResult):
    category: str = Field(description="bug | feature | billing")


class ResolveParams(BaseParams):
    ticket_id: str = Field(description="Ticket identifier")
    category: str = Field(description="Ticket category")


class ResolveResult(BaseResult):
    resolved: bool = Field(description="Resolved flag")
    note: str = Field(description="Resolution note")

## Actions and plain function node

Action `Params` fields are extracted from the agent state by name automatically.
`Result` fields are merged back into the state after the node returns.
Plain async functions receive the full state dict.

In [ ]:
@meta(description="Classify a support ticket by content", domain=SupportDomain)
@check_roles(GuestRole)
class ClassifyTicketAction(BaseAction[ClassifyParams, ClassifyResult]):

    @summary_aspect("Classify ticket")
    async def classify(self, params, state, box, connections):
        text = params.note.lower()
        if "billing" in text or "invoice" in text or "payment" in text:
            cat = "billing"
        elif "crash" in text or "error" in text or "bug" in text:
            cat = "bug"
        else:
            cat = "feature"
        return ClassifyResult(category=cat)


@meta(description="Route ticket to engineering team", domain=SupportDomain)
@check_roles(GuestRole)
class EngineeringAction(BaseAction[ResolveParams, ResolveResult]):

    @summary_aspect("Assign to engineering")
    async def resolve(self, params, state, box, connections):
        tag = params.category.upper()
        return ResolveResult(
            resolved=True,
            note=f"[{tag}] #{params.ticket_id} → engineering",
        )


@meta(description="Route ticket to billing team", domain=SupportDomain)
@check_roles(GuestRole)
class BillingAction(BaseAction[ResolveParams, ResolveResult]):

    @summary_aspect("Assign to billing")
    async def resolve(self, params, state, box, connections):
        return ResolveResult(
            resolved=True,
            note=f"[BILLING] #{params.ticket_id} → billing team",
        )


async def close_ticket(state: TicketState) -> dict:
    return {"note": state["note"] + " [closed]"}

## Run

> In Colab, `await` works at top level — no `asyncio.run()`.

In [ ]:
async def main() -> None:
    machine = ActionProductMachine()
    context = Context()

    # --- Part 1: early error detection ---
    # .edge() validates that both endpoints are registered. A typo raises
    # UnregisteredNodeError immediately, before ainvoke() is ever called.
    try:
        (
            LangGraphAdapter(machine=machine, context=context, agentstate=TicketState)
            .node(ClassifyTicketAction())
            .edge(ClassifyTicketAction, "typo_node")  # not registered
        )
    except UnregisteredNodeError as exc:
        print(f"[UnregisteredNodeError] {exc}\n")

    # --- Part 2: multi-path routing with .route() ---
    # .route(source, on=fn, paths={key: target}) routes to a different node
    # based on a function that returns a key from the current state.
    compiled = (
        LangGraphAdapter(machine=machine, context=context, agentstate=TicketState)
        .node(ClassifyTicketAction())
        .node(EngineeringAction())
        .node(BillingAction())
        .node(close_ticket, name="close_ticket")
        .start(ClassifyTicketAction)
        .route(
            ClassifyTicketAction,
            on=lambda s: s.get("category"),
            paths={
                "bug":     EngineeringAction,
                "feature": EngineeringAction,
                "billing": BillingAction,
            },
        )
        .edge(EngineeringAction, "close_ticket")
        .edge(BillingAction,     "close_ticket")
        .edge("close_ticket",    END)
        .compile()
    )

    tickets = [
        {"ticket_id": "T-1", "note": "App crashes on startup",     "category": "", "resolved": False},
        {"ticket_id": "T-2", "note": "Wrong invoice amount",       "category": "", "resolved": False},
        {"ticket_id": "T-3", "note": "Add support for dark mode",  "category": "", "resolved": False},
        {"ticket_id": "T-4", "note": "Payment failed on checkout", "category": "", "resolved": False},
    ]
    print("=== Ticket routing ===")
    for t in tickets:
        r = await compiled.ainvoke(t)
        print(f"  {r['ticket_id']}  [{r['category']:8}]  {r['note']}")

    # --- Part 3: native continuation via build_graph() ---
    # build_graph() returns the StateGraph before compilation.
    # Add more nodes/edges with the native LangGraph API, then compile.
    print("\n=== Native continuation ===")
    graph = (
        LangGraphAdapter(machine=machine, context=context, agentstate=TicketState)
        .node(ClassifyTicketAction())
        .start(ClassifyTicketAction)
        .build_graph()
    )
    graph.add_edge("classify_ticket", END)
    native_compiled = graph.compile()
    r = await native_compiled.ainvoke(
        {"ticket_id": "T-5", "note": "billing question", "category": "", "resolved": False}
    )
    print(f"  T-5 classified as: {r['category']}")

    print("\n=== Graph (Mermaid) ===")
    print(compiled.get_graph().draw_mermaid())


await main()